# HydraPTA: Model Validation & Petroleum Engineering Verification

This Jupyter Notebook provides end-to-end technical validation and verification for **HydraPTA** (Hydraulic Fracture Pressure Transient Analyzer).

### Objectives:
1. **Model Setup & Petrophysical Definitions**
2. **Forward Transient Pressure Simulation** via Laplace Solution & Stehfest Inversion
3. **Bourdet Logarithmic Derivative Calculation**
4. **Diagnostic Flow Regime Verification** (Wellbore storage, Bilinear flow, Formation linear flow, Pseudoradial flow)
5. **Parameter Sensitivity Sweeps** ($x_f, k_f, w_f, C_f, k, q$)
6. **Inverse Fracture Characterization & Synthetic Validation** ($x_f, C_f$ estimation without ground-truth leakage)
7. **Verification of All 3 Predefined Educational Cases**

In [ ]:
import sys
import os
sys.path.append('..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go

from src.models import ReservoirProperties, WellProperties, FractureProperties
from src.fracture import calculate_fracture_conductivity, calculate_dimensionless_conductivity, classify_fracture_conductivity_type
from src.pressure_transient import generate_synthetic_transient_response, stehfest_weights
from src.derivative import calculate_bourdet_derivative, calculate_log_slopes
from src.flow_regime import detect_sustained_flow_regimes, generate_flow_regime_summary_table
from src.estimation import run_inverse_characterization
from src.sensitivity import generate_sensitivity_scenarios
from data.demo_cases import DEMO_CASES

print("HydraPTA validation environment initialized successfully.")

## 1. Demo Case 1: High-Conductivity Hydraulic Fracture

In [ ]:
case1 = DEMO_CASES["case_1"]
res1 = case1["reservoir"]
well1 = case1["well"]
frac1 = case1["fracture"]

cf1 = frac1.Cf
cfd1 = frac1.CfD(res1.k)
cat1, desc1 = classify_fracture_conductivity_type(cfd1)

print(f"Case 1: {case1['name']}")
print(f"Fracture Half-Length (xf): {frac1.xf} ft")
print(f"Fracture Permeability (kf): {frac1.kf} mD, Width (wf): {frac1.wf} ft")
print(f"Fracture Conductivity (Cf): {cf1} mD-ft")
print(f"Dimensionless Conductivity (CfD): {cfd1:.4f}")
print(f"Classification: {cat1}")

In [ ]:
# Simulate Case 1
df1 = generate_synthetic_transient_response(res1, well1, frac1, n_points=160)
t1 = df1["time_hr"].values
dp1 = df1["delta_P_psi"].values

# Bourdet derivative & slopes
derv1 = calculate_bourdet_derivative(t1, dp1, L=0.15)
slopes1 = calculate_log_slopes(t1, derv1, L=0.20)

regimes1 = detect_sustained_flow_regimes(t1, dp1, derv1, slopes1)
print("Detected Flow Regimes for Case 1:")
print(generate_flow_regime_summary_table(regimes1))

## 2. Inverse Parameter Characterization Validation (No Data Leakage)

In [ ]:
# Inverse characterization
val1, meta1 = run_inverse_characterization(t1, dp1, derv1, regimes1, res1, well1, frac1)

print("Synthetic-Data Validation Table (Case 1):")
for v in val1:
    est = f"{v.estimated_value} {v.unit}" if v.estimated_value is not None else "Not reliably estimated"
    err = f"{v.relative_error:.2f}%" if v.relative_error is not None else "—"
    print(f"{v.parameter:35s} | True: {v.true_value:8.1f} {v.unit:6s} | Est: {est:20s} | Rel Err: {err:8s} | Status: {v.status_message}")

## 3. Demo Case 2: Finite-Conductivity Fracture (Bilinear Flow Signature)

In [ ]:
case2 = DEMO_CASES["case_2"]
res2 = case2["reservoir"]
well2 = case2["well"]
frac2 = case2["fracture"]

df2 = generate_synthetic_transient_response(res2, well2, frac2, n_points=160)
t2 = df2["time_hr"].values
dp2 = df2["delta_P_psi"].values
derv2 = calculate_bourdet_derivative(t2, dp2, L=0.15)
slopes2 = calculate_log_slopes(t2, derv2, L=0.20)

regimes2 = detect_sustained_flow_regimes(t2, dp2, derv2, slopes2)
print("Detected Flow Regimes for Case 2 (Finite Conductivity):")
print(generate_flow_regime_summary_table(regimes2))

val2, meta2 = run_inverse_characterization(t2, dp2, derv2, regimes2, res2, well2, frac2)
print("\nSynthetic-Data Validation Table (Case 2):")
for v in val2:
    est = f"{v.estimated_value} {v.unit}" if v.estimated_value is not None else "Not reliably estimated"
    err = f"{v.relative_error:.2f}%" if v.relative_error is not None else "—"
    print(f"{v.parameter:35s} | True: {v.true_value:8.1f} {v.unit:6s} | Est: {est:20s} | Rel Err: {err:8s} | Status: {v.status_message}")

## 4. Demo Case 3: Long-Fracture Penetration ($x_f = 1000$ ft)

In [ ]:
case3 = DEMO_CASES["case_3"]
res3 = case3["reservoir"]
well3 = case3["well"]
frac3 = case3["fracture"]

df3 = generate_synthetic_transient_response(res3, well3, frac3, n_points=160)
t3 = df3["time_hr"].values
dp3 = df3["delta_P_psi"].values
derv3 = calculate_bourdet_derivative(t3, dp3, L=0.15)
slopes3 = calculate_log_slopes(t3, derv3, L=0.20)

regimes3 = detect_sustained_flow_regimes(t3, dp3, derv3, slopes3)
print("Detected Flow Regimes for Case 3 (Long Fracture):")
print(generate_flow_regime_summary_table(regimes3))

val3, meta3 = run_inverse_characterization(t3, dp3, derv3, regimes3, res3, well3, frac3)
print("\nSynthetic-Data Validation Table (Case 3):")
for v in val3:
    est = f"{v.estimated_value} {v.unit}" if v.estimated_value is not None else "Not reliably estimated"
    err = f"{v.relative_error:.2f}%" if v.relative_error is not None else "—"
    print(f"{v.parameter:35s} | True: {v.true_value:8.1f} {v.unit:6s} | Est: {est:20s} | Rel Err: {err:8s} | Status: {v.status_message}")

## 5. Parameter Sensitivity Sweeps & Interpretation Verification

In [ ]:
df_sens_curves, df_sens_summary, insight = generate_sensitivity_scenarios(
    param_key="xf",
    base_reservoir=res1,
    base_well=well1,
    base_fracture=frac1
)

print("Sensitivity Summary Table (Fracture Half-Length xf):")
print(df_sens_summary.to_string())
print("\nGenerated Engineering Insight:")
print(insight)